# MAGDA Fine-Tuning: Qwen 2.5 Coder 7B

Fine-tunes a LoRA adapter for three tasks:
- **Router**: classify intent (COMMAND / MUSIC / BOTH)
- **Command**: generate MAGDA DSL code for DAW operations
- **Music**: generate chord progressions, notes, arpeggios in compact notation

Requirements: Colab Pro (A100 GPU recommended, T4 works too)

In [ ]:
# Step 1: Install dependencies
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
# Step 2: Mount Google Drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

drive_dir = '/content/drive/MyDrive/magda-training'
os.makedirs(drive_dir, exist_ok=True)

# Upload dataset.jsonl if not already on Drive
dataset_drive = os.path.join(drive_dir, 'dataset.jsonl')
if os.path.exists(dataset_drive):
    shutil.copy2(dataset_drive, 'dataset.jsonl')
    print(f"Loaded dataset.jsonl from Drive")
elif os.path.exists('dataset.jsonl'):
    shutil.copy2('dataset.jsonl', dataset_drive)
    print("dataset.jsonl copied to Drive for next time")
else:
    from google.colab import files
    uploaded = files.upload()  # select dataset.jsonl
    shutil.copy2('dataset.jsonl', dataset_drive)
    print("dataset.jsonl uploaded and backed up to Drive")

In [ ]:
# Step 3: Load and inspect dataset
import json
from datasets import Dataset

with open('dataset.jsonl', 'r') as f:
    raw_data = [json.loads(line) for line in f]

print(f"Total examples: {len(raw_data)}")
print(f"Example: {json.dumps(raw_data[0], indent=2)[:300]}")

In [ ]:
# Step 4: Load base model with unsloth (4-bit quantized for training)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=2048,
    dtype=None,       # auto-detect
    load_in_4bit=True,
)

print(f"Model loaded: {model.config._name_or_path}")

In [ ]:
# Step 5: Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                    # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,          # optimized = 0
    bias="none",
    use_gradient_checkpointing="unsloth",  # saves VRAM
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
# Step 6: Format dataset for Qwen chat template

def format_chat(example):
    """Convert messages list to Qwen ChatML format."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_chat)

# Preview a formatted example
print(dataset[0]["text"][:500])
print("---")
print(dataset[-1]["text"][:500])

In [ ]:
# Step 7: Train
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=True,  # pack short examples together for efficiency
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=not __import__('torch').cuda.is_bf16_supported(),
        bf16=__import__('torch').cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="magda-lora-output",
        report_to="none",
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"Training complete! Loss: {stats.training_loss:.4f}")

## Step 8: Validation

In-notebook inference with Unsloth's 4-bit quantized model is unreliable (RoPE shape bugs in the optimized attention path). **Skip to Step 9** — validate after GGUF export by running the model in MAGDA or with `llama-server`.

In [ ]:
# Step 9: Save LoRA adapter
model.save_pretrained("magda-lora")
tokenizer.save_pretrained("magda-lora")
print("LoRA adapter saved to magda-lora/")

In [ ]:
# Step 10: Free disk space, then merge + export GGUF directly to Google Drive
import shutil, os

# Clean up training checkpoints to reclaim space
for d in ["magda-lora-output", "magda-merged", "magda-gguf"]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Deleted {d}/")

total, used, free = shutil.disk_usage("/")
print(f"Free disk: {free / (1024**3):.1f} GB")

# Export GGUF (merge + convert + quantize in one call)
model.save_pretrained_gguf(
    "magda-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

# Copy GGUF to Google Drive
import glob
gguf_files = glob.glob("**/*.gguf", recursive=True)
if gguf_files:
    drive_models = '/content/drive/MyDrive/magda-training'
    os.makedirs(drive_models, exist_ok=True)
    dest = os.path.join(drive_models, os.path.basename(gguf_files[0]))
    print(f"Copying to Google Drive...")
    shutil.copy2(gguf_files[0], dest)
    print(f"GGUF saved to Drive: {dest}")
    print("Download from drive.google.com on your Mac.")
else:
    print("No GGUF file found!")
    !find . -name "*.gguf" 2>/dev/null

## Done!

Copy the downloaded `.gguf` file to your models directory and load it in MAGDA's AI Settings dialog.